In [23]:
from __future__ import annotations

import pickle
import re,json
from pathlib import Path

import automated_intelligence_tests as ait
from tqdm import tqdm

DATA_ROOT = Path("data")
TASKS = ("dat", "aut", "cwt")

print("ait", getattr(ait, "__version__", "?"))

_NUM = re.compile(r"^\s*[\(\[]?\d+[.)\]]\s*|^\s*[-*•]\s*")
_WRAP = re.compile(r"^[\s\"'`“”‘’\[\(]+|[\s\"'`“”‘’\]\)]+$")

ait 0.1.7


In [18]:
FENCE_RE = re.compile(r"```(?:[a-zA-Z0-9_+-]+)?\s*\n?(.*?)\n?```", flags=re.S)
UNCLOSED_FENCE_RE = re.compile(r"```(?:[a-zA-Z0-9_+-]+)?\s*\n?(.*)$", flags=re.S)
THINK_RE = re.compile(
    r"<(?:think|thinking|reasoning)>.*?</(?:think|thinking|reasoning)>",
    flags=re.S | re.I,)
ITEM_LABEL_RE = re.compile(
    r"""^\s*(?:
        \d+\s*[\.\)\:] |
        [-*•] |
        (?:word|noun|item)[_\s-]?\d+\s*[:=] |
        \d{1,2}(?=[A-Za-z])
    )\s*""",
    flags=re.I | re.X,)
PREAMBLE_HINT = re.compile(
    r"""^(?:
        (?:sure|certainly|of\ course|okay|ok|alright|right|yes|yeah|got\ it)[!.,]?\s*
      | (?:here|this|these|the|my|your)
      | (?:nouns?|words?|items?|list|answer|response|output)
      | (?:i(?:'|’)ll|i\ will|let\ me)
    )""",
    flags=re.I | re.X,)
LABEL_ONLY_LINE = re.compile(
    r"""^\s*(?:
        (?:sure|certainly|of\ course|okay|ok|alright|right|yes|yeah|got\ it)[!.,]*
      | (?:here(?:'|’)s|here\ is|here\ are|here\ they\ are|here\ you\ (?:go|are))
          (?:\s+\w+){0,8}
      | (?:the\ )?(?:(?:ten|10|following|requested|distinct)\s+)?
          (?:nouns?|words?|items?|list)
          (?:\s+(?:are|is|follow(?:s|ing)?))?
    )\s*:?\s*$""",
    flags=re.I | re.X,)
JSON_WORD_RE = re.compile(
    r"""["']?(?:word|noun|item)[_\s-]?(\d+)["']?\s*[:=]\s*["']([^"']+)["']""",
    flags=re.I,)
AUT_ITEM_LABEL = re.compile(
    r"""^\s*(?:
        [\(\[]?\d+[.)\]:\-–—] |
        [-*•] |
        (?:word|noun|item|use)[_\-\s]?\d+\s*[:=]
    )\s*""",re.I | re.X,)

def unwrap_fences(text):
    if "```" not in text:
        return text
    chunks = [m.group(1).strip() for m in FENCE_RE.finditer(text) if m.group(1).strip()]
    if chunks:
        return "\n".join(chunks)
    m = UNCLOSED_FENCE_RE.search(text)
    return m.group(1).strip() if m else text.replace("```", " ")

def strip_preamble(text):
    text = text.strip().strip("\"'`")
    while True:
        lines = text.splitlines()
        if not lines or not LABEL_ONLY_LINE.match(lines[0]):
            break
        text = "\n".join(lines[1:]).strip()
    # First colon only, and only when the left-hand side is a lead-in.
    if ":" in text:
        left, right = text.split(":", 1)
        left_s = left.strip()
        if (
            right.strip()
            and len(left_s) <= 80
            and not ITEM_LABEL_RE.match(left_s + ":")
            and PREAMBLE_HINT.match(left_s)
        ):
            text = right.strip()
    return text.strip()

def parse_dat(raw, n=10):
    text = str(raw or "").strip()
    if not text:
        return None
    text = THINK_RE.sub(" ", text)
    text = unwrap_fences(text)

    pairs = JSON_WORD_RE.findall(text)
    if len(pairs) >= 3:
        tokens = [p[1].strip() for p in sorted(pairs, key=lambda p: int(p[0]))]
    else:
        text = strip_preamble(text)
        tokens = []
        for part in re.split(r"[,;\n\r]+", text):
            part = ITEM_LABEL_RE.sub("", part).strip().strip("\"'`")
            if part:
                tokens.append(part)
        # space-separated blob with no commas ("apple banana volcano …")
        if len(tokens) == 1 and " " in tokens[0]:
            words = tokens[0].split()
            if 7 <= len(words) <= 12:
                tokens = words

    nouns, seen = [], set()
    for t in tokens:
        t = t.strip()
        if t.count(" ") >= 3 or (t.endswith(".") and " " in t):
            continue  # commentary, not a noun / short compound
        key = t.lower()
        if not t or key in seen:
            continue
        seen.add(key)
        nouns.append(t)
        if len(nouns) == n:
            break
            
    if not nouns:
        return None
    return (nouns + [None] * 10)[:10]


def score_dat(parsed, stim=None):
    """parsed list -> DAT score (mean pairwise distance x 100) or None.

    Calls ait.evaluate("dat", list) unchanged.
    Score is None when fewer than 7 unique valid words.
    DAT needs no cue.
    """
    if not parsed:
        return None
    try:
        result = ait.evaluate("dat", list(parsed))
    except Exception as e:
        print("score_dat failed:", e)
        return None
    if not result:
        return None
    return result.get("score")

In [40]:
fps = sorted(p for p in Path('./data/').joinpath('dat').rglob("*.pickle") if p.is_file())

n=0
for p in fps:
    if n<4:
        with p.open("rb") as f:
            row = pickle.load(f)
        if row.get('raw')!='':
            print(parse_dat(row.get('raw')))
            print(score_dat(parse_dat(row.get('raw'))))
            n+=1
    else:
        break

['elephant', 'whisper', 'gravity', 'joy', 'rust', 'ocean', 'shadow', 'mathematics', 'rebellion', 'silence']
75.18111158694539
['serendipity', 'anvil', 'whisper', 'glacier', 'carousel', 'nostalgia', 'anchor', 'phosphorescence', 'labyrinth', 'custard']
85.04310807301884
['telescope', 'melody', 'shadow', 'courage', 'tornado', 'marble', 'laughter', 'algorithm', 'fossil', 'butter']
83.9017758312236
['telescope', 'butterfly', 'democracy', 'thunder', 'sandwich', 'shadow', 'melody', 'brick', 'curiosity', 'ocean']
84.19093664150361


In [47]:
def parse_aut(raw):
    text = THINK_RE.sub(" ", str(raw or ""))
    text = re.sub(r"<br\s*/?>", "\n", text, flags=re.I)
    if "```" in text:
        text = FENCE_RE.sub(lambda m: "\n" + (m.group(1) or "").strip() + "\n", text)
        text = UNCLOSED_FENCE_RE.sub(lambda m: "\n" + (m.group(1) or "").strip(), text)
        text = text.replace("```", " ")
    text = re.sub(r"^#+\s*.*$", "", text, flags=re.M)
    text = text.strip().strip("\"'`“”‘’")
    if not text:
        return None
    chunks = re.split(r"[\n\r;]+", text)
    if len(chunks) == 1 and chunks[0].count(",") >= 3:
        chunks = re.split(r",\s*", chunks[0])
    uses = []
    for chunk in chunks:
        chunk = AUT_ITEM_LABEL.sub("", chunk)
        chunk = chunk.strip().strip("\"'`“”‘’")
        chunk = re.sub(r"\s+", " ", chunk)
        if len(chunk) < 2:
            continue
        if LABEL_ONLY_LINE.match(chunk):
            continue
        low = chunk.lower()
        if low.startswith(("here are", "here is", "creative uses")):
            continue
        if low in {"uses", "use"}:
            continue
        uses.append(chunk)
    return uses or None


def score_aut(parsed, stim=None):
    """parsed uses + stim cue -> AUT SemDis score or None.

    Wraps ait.evaluate("aut", {"cue", "responses"}) unchanged.
    Cue must come from pickle["kwargs"]["cue"]. Never invent a cue.
    """
    if not parsed:
        return None
    stim = stim or {}
    cue = stim.get("cue") if isinstance(stim, dict) else stim
    if isinstance(cue, (list, tuple)):
        cue = " ".join(str(x) for x in cue if str(x).strip())
    if not cue:
        return None
    try:
        result = ait.evaluate("aut", {"cue": cue, "responses": list(parsed)})
    except Exception as e:
        print("score_aut failed:", e)
        return None
    if not result:
        return None
    return result.get("score")

In [48]:
fps = sorted(p for p in Path("./data/").joinpath("aut").rglob("*.pickle") if p.is_file())
n = 0
for p in fps:
    with p.open("rb") as f:
        row = pickle.load(f)
    raw = row.get("raw") or ""
    if not raw:
        continue
    stim = row.get("kwargs") or {}
    parsed = parse_aut(raw)
    print("file:", p)
    print("cue:", stim.get("cue"))
    print("parsed:", parsed)
    print("score:", score_aut(parsed, stim))
    n += 1
    if n >= 4:
        break

file: data/aut/claude-haiku-4.5/0.5/0009c4e719194d81.pickle
cue: brick
parsed: ['Doorstop for windy days', 'Paperweight for important documents', 'Bookend for leaning stacks', 'Makeshift hammer for gentle tasks', 'Garden edging for flower beds', 'Stepping stone in muddy areas', 'Weight for securing tarps', 'Anchor for clothesline', 'Platform for potted plants', 'Paperweight for outdoor tables', 'Yoga block for stretching', 'Doorbell weight (tied to string)', 'Planter for succulents', 'Marking stone for garden rows', 'Improvised bench for short breaks', 'Weight for exercise routines', 'Paperweight for art projects', 'Base for bird feeder pole', 'Doormat weight in hallways', 'Sculpture material for art installations', 'Bookshelf spacer', 'Threshold ramp for small pets', 'Weight for securing garden netting', 'Makeshift level checker (flat surface)', 'Decorative garden accent', 'Fireplace prop for ambiance', 'Cooling stone for sore muscles', 'Weight for pressing flowers', 'Stacking toy for

In [50]:
def parse_cwt(raw):
    """Raw CWT text -> [story] (one-element list), or None.

    Johnson et al. 2023 format extraction only.
    Wrapped in a list so parsed is list|None for every test.
    Lifts THINK_RE + FENCE unwrap (keep inner) from DAT.
    Does not lift ITEM_LABEL / PREAMBLE_HINT / LABEL_ONLY_LINE / JSON_WORD
    (a story sentence can start with "Here is …").
    """
    text = THINK_RE.sub(" ", str(raw or ""))
    if "```" in text:
        m = FENCE_RE.search(text)
        if m and m.group(1).strip():
            text = m.group(1)
        else:
            m = UNCLOSED_FENCE_RE.search(text)
            if m and m.group(1).strip():
                text = m.group(1)
    text = re.sub(r"^#+\s*.*$", "", text, flags=re.M)
    text = re.sub(r"^\s*Title:.*$", "", text, flags=re.M | re.I)
    text = re.sub(
        r"^\s*(Here(?:'|’)s (?:a |the )?story|Story)\s*:\s*",
        "",
        text,
        flags=re.I,
    )
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return [text] if text else None


def score_cwt(parsed, stim=None):
    """parsed [story] + stim cue -> CWT BERT DSI score or None.

    Unwraps parsed[0]. Cue is kwargs["cue"] as stored (list or string).
    First call downloads bert-large.
    """
    if not parsed:
        return None
    story = parsed[0] if isinstance(parsed, list) else parsed
    if not str(story).strip():
        return None
    stim = stim or {}
    cue = stim.get("cue") if isinstance(stim, dict) else stim
    if cue is None:
        cue = []
    if isinstance(cue, str):
        cue = [w for w in re.split(r"[,\s]+", cue) if w]
    try:
        result = ait.evaluate("cwt", {"cue": list(cue), "story": str(story)})
    except Exception as e:
        print("score_cwt failed:", e)
        return None
    if not result:
        return None
    return result.get("score")

In [53]:
fps = sorted(p for p in Path("./data/").joinpath("cwt").rglob("*.pickle") if p.is_file())
n = 0
for p in fps:
    with p.open("rb") as f:
        row = pickle.load(f)
    raw = row.get("raw") or ""
    if not raw:
        continue
    stim = row.get("kwargs") or {}
    parsed = parse_cwt(raw)
    print("file:", p)
    print("cue:", stim.get("cue"))
    print("parsed:", parsed)
    print("score:", score_cwt(parsed, stim))
    n += 1
    if n >= 3:
        break

file: data/cwt/claude-haiku-4.5/0.5/0065770d1b2b7cc3.pickle
cue: ['t', 'i', 't', 'l', 'e', ' ', '(', '2', '3', '0', '5', ' ', 'o', 'r', ' ', 'E', 'x', 'e', 'c', 'u', 't', 'i', 'o', 'n', ')']
parsed: ['In the year 2305, the title of "Executioner" had become nothing more than a historical curiosity, a relic of humanity\'s darker past that museums preserved behind glass. Dr. Elena Torres stood in the archive, examining the ancient execution device with trembling fingers, wondering what drove people to such cruelty centuries before her time. The machine\'s cold metal seemed to whisper stories of injustice and fear, each scratch and dent a testament to lives ended by state decree. She realized that studying this artifact wasn\'t just about understanding history—it was about ensuring such horrors could never return to plague civilization again. As she documented her findings, Elena felt the weight of responsibility settle upon her shoulders, knowing that remembrance itself was the truest for

In [ ]:
PARSE_FN = {"dat": parse_dat, "aut": parse_aut, "cwt": parse_cwt}
SCORE_FN = {"dat": score_dat, "aut": score_aut, "cwt": score_cwt}

def backfill_task(task, data_root=DATA_ROOT):
    task = task.strip().lower()
    parse_fn = PARSE_FN[task]
    score_fn = SCORE_FN[task]
    fps = sorted(p for p in Path(data_root).joinpath(task).rglob("*.pickle") if p.is_file())
    for p in tqdm(fps, desc=f"backfill {task}"):
        try:
            with p.open("rb") as f:
                row = pickle.load(f)
            parsed = parse_fn(row.get("raw"))
            score = score_fn(parsed, row.get("kwargs") or {})
            row["parsed"] = parsed
            row["score"] = score
            tmp = p.with_name(p.name + ".tmp")
            with tmp.open("wb") as f:
                pickle.dump(row, f, protocol=pickle.HIGHEST_PROTOCOL)
            tmp.replace(p)
        except Exception as e:
            print("SKIP", p, e)